# 📊 AI-Assisted Trading Risk Manager
**Sentiment → Risk Models → Portfolio Optimisation → Trailing Stops → Monte Carlo**

**Install dependencies:**
```bash
pip install yfinance transformers torch numpy scipy plotly arch hmmlearn feedparser
```


In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# ─── PORTFOLIO CONFIG ────────────────────────────────────────────────────────
TICKERS   = ['AAPL', 'MSFT', 'XOM', 'GS', 'JPM']
START     = '2020-01-01'
END       = '2024-12-31'
RISK_FREE = 0.05            # annual risk-free rate

# ─── MONTE CARLO CONFIG ──────────────────────────────────────────────────────
HORIZON  = 252              # trading days (~1 year)
N_PATHS  = 10_000
PORT_VAL = 1_000_000        # starting portfolio value ($)

print('✅ Imports OK')


✅ Imports OK


---
## Phase 1 — Financial Sentiment Engine (FinBERT + live news)
Fetches real headlines from Yahoo Finance for each ticker via `yfinance`.
Falls back to a curated sample set if the network is unavailable.


In [2]:
from transformers import pipeline

# ── 1a. Fetch real headlines ──────────────────────────────────────────────
MAX_HEADLINES_PER_TICKER = 6   # adjust as needed

def get_ticker_headlines(tickers, max_per=MAX_HEADLINES_PER_TICKER):
    """
    Fetch recent news headlines via yfinance.
    Returns list of {'ticker': str, 'headline': str}.
    Handles both old (item['title']) and new (item['content']['title']) API formats.
    """
    results = []
    for t in tickers:
        try:
            news = yf.Ticker(t).news or []
            for item in news[:max_per]:
                # new yfinance format
                if 'content' in item and isinstance(item['content'], dict):
                    title = item['content'].get('title', '')
                else:
                    title = item.get('title', '')
                if title:
                    results.append({'ticker': t, 'headline': title})
        except Exception as e:
            print(f'  ⚠️  Could not fetch news for {t}: {e}')
    return results

print('Fetching live headlines...')
live_items = get_ticker_headlines(TICKERS)

# ── Fallback sample set (used only if no live data) ───────────────────────
FALLBACK_HEADLINES = [
    {'ticker': 'AAPL', 'headline': 'Apple beats earnings expectations, raises guidance'},
    {'ticker': 'AAPL', 'headline': 'iPhone 16 demand stronger than anticipated in Asia'},
    {'ticker': 'MSFT', 'headline': 'Microsoft Azure cloud revenue surges 33% year-on-year'},
    {'ticker': 'MSFT', 'headline': 'Goldman Sachs upgrades Microsoft to strong buy'},
    {'ticker': 'XOM',  'headline': 'Oil prices tumble on demand fears amid global slowdown'},
    {'ticker': 'XOM',  'headline': 'ExxonMobil raises dividend as oil profits remain elevated'},
    {'ticker': 'GS',   'headline': 'Goldman Sachs Q3 profit rises on trading revenue rebound'},
    {'ticker': 'GS',   'headline': 'Wall Street banks face tighter capital requirements'},
    {'ticker': 'JPM',  'headline': 'JPMorgan warns of credit losses in commercial real estate'},
    {'ticker': 'JPM',  'headline': 'Fed signals two more rate hikes this year'},
]

if live_items:
    print(f'✅ Fetched {len(live_items)} live headlines across {len(TICKERS)} tickers.')
    headline_items = live_items
else:
    print('⚠️  No live headlines retrieved — using fallback sample set.')
    headline_items = FALLBACK_HEADLINES

headlines = [h['headline'] for h in headline_items]

# ── 1b. Run FinBERT ───────────────────────────────────────────────────────
print('\nLoading FinBERT (downloads ~420 MB on first run)...')
sentiment_pipe = pipeline(
    'text-classification',
    model='ProsusAI/finbert',
    top_k=None   # replaces deprecated return_all_scores=True
)

results = sentiment_pipe(headlines)

rows = []
for item, scores in zip(headline_items, results):
    if isinstance(scores, dict):
        scores = [scores]
    score_dict = {s['label']: s['score'] for s in scores}
    dominant   = max(score_dict, key=score_dict.get)
    rows.append({
        'ticker':    item['ticker'],
        'headline':  item['headline'],
        'positive':  round(score_dict.get('positive', 0), 3),
        'negative':  round(score_dict.get('negative', 0), 3),
        'neutral':   round(score_dict.get('neutral',  0), 3),
        'sentiment': dominant,
    })

sentiment_df = pd.DataFrame(rows)
display(sentiment_df)

# ── 1c. Per-ticker & portfolio aggregates ────────────────────────────────
ticker_sentiment = (
    sentiment_df.groupby('ticker')
    .apply(lambda g: g['positive'].mean() - g['negative'].mean())
    .rename('score')
    .round(3)
)
portfolio_sentiment = ticker_sentiment.mean()

print('\n📊 Per-ticker sentiment:')
print(ticker_sentiment.to_string())
print(f'\n📰 Portfolio sentiment score: {portfolio_sentiment:+.3f}')
print('    (range: -1 = very bearish → +1 = very bullish)')


Fetching live headlines...
✅ Fetched 30 live headlines across 5 tickers.

Loading FinBERT (downloads ~420 MB on first run)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13484.57it/s]


,ticker,headline,positive,negative,neutral,sentiment
0,AAPL,Is Intel’s (INTC) Confidential AI Push Quietly...,0.031,0.257,0.712,neutral
1,AAPL,Apple Inc. (AAPL)’s Durable Growth Narrative K...,0.933,0.013,0.054,positive
2,AAPL,Why American Express Is Still a Top Buffett St...,0.060,0.025,0.915,neutral
3,AAPL,CDL’s $2.29 annual dividend beats Treasury yie...,0.942,0.029,0.029,positive
4,AAPL,"Trump’s 3,711 Trades Point to Multiple Stock-M...",0.029,0.030,0.940,neutral
5,AAPL,2 S&P 500 Stocks to Research Further and 1 We ...,0.072,0.014,0.914,neutral
6,MSFT,AI trade: Investors may want to look outside i...,0.064,0.015,0.921,neutral
7,MSFT,Microsoft Maia Chip Talks With Anthropic Test ...,0.288,0.008,0.703,neutral
8,MSFT,The 401(k) Mega Backdoor Roth Strategy a Tech ...,0.052,0.017,0.931,neutral
9,MSFT,CDL’s $2.29 annual dividend beats Treasury yie...,0.942,0.029,0.029,positive



📊 Per-ticker sentiment:
ticker
AAPL    0.283
GS      0.121
JPM    -0.454
MSFT    0.373
XOM     0.294

📰 Portfolio sentiment score: +0.123
    (range: -1 = very bearish → +1 = very bullish)


---
## Phase 1b — Trailing Stop Configuration
Each trade is split into **3 tranches** (⅓ each).  
A trailing stop watches the **drawdown from the running peak** of the in-market value.  
When the drawdown breaches a level, that tranche is locked in as cash — the rest keeps running.

You can either **enter your own levels** (`USE_MANUAL = True`) or let the notebook  
**auto-suggest levels from the sentiment score** above.


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# TRAILING STOP CONFIG
# Set USE_MANUAL = True and fill MANUAL_STOPS to use your own levels.
# Each stop: level = drawdown threshold (negative %), exit_fraction = % of portfolio to lock in.
# ─────────────────────────────────────────────────────────────────────────────

USE_MANUAL = False          # ← flip to True to override sentiment-based suggestions

MANUAL_STOPS = [
    {'level': -0.05, 'exit_fraction': 1/3, 'label': 'Stop 1 (-5%)'},
    {'level': -0.10, 'exit_fraction': 1/3, 'label': 'Stop 2 (-10%)'},
    {'level': -0.15, 'exit_fraction': 1/3, 'label': 'Stop 3 (-15%)'},
]

# ── Auto-suggest from sentiment ───────────────────────────────────────────
def sentiment_to_stops(score):
    """
    Sentiment score -1…+1.
    Bullish  → wider stops (give the trade more room to breathe).
    Bearish  → tighter stops (protect capital more aggressively).
    """
    if score >= 0.3:       # bullish
        return [
            {'level': -0.08, 'exit_fraction': 1/3, 'label': 'Stop 1 (-8%)'},
            {'level': -0.14, 'exit_fraction': 1/3, 'label': 'Stop 2 (-14%)'},
            {'level': -0.20, 'exit_fraction': 1/3, 'label': 'Stop 3 (-20%)'},
        ]
    elif score <= -0.3:    # bearish
        return [
            {'level': -0.03, 'exit_fraction': 1/3, 'label': 'Stop 1 (-3%)'},
            {'level': -0.06, 'exit_fraction': 1/3, 'label': 'Stop 2 (-6%)'},
            {'level': -0.10, 'exit_fraction': 1/3, 'label': 'Stop 3 (-10%)'},
        ]
    else:                  # neutral
        return [
            {'level': -0.05, 'exit_fraction': 1/3, 'label': 'Stop 1 (-5%)'},
            {'level': -0.10, 'exit_fraction': 1/3, 'label': 'Stop 2 (-10%)'},
            {'level': -0.15, 'exit_fraction': 1/3, 'label': 'Stop 3 (-15%)'},
        ]

STOPS = MANUAL_STOPS if USE_MANUAL else sentiment_to_stops(portfolio_sentiment)

print(f'{"⚙️  Manual stops" if USE_MANUAL else "🤖 Sentiment-derived stops"} (score={portfolio_sentiment:+.3f}):')
print(f'  {"Level":>12}   Exit fraction   Label')
print('  ' + '-'*45)
for s in STOPS:
    print(f'  {s["level"]:>+11.1%}   {s["exit_fraction"]:>12.1%}   {s["label"]}')
print('\nEach stop exits that fraction of the REMAINING in-market value.')
print('The rest of the portfolio continues to follow the simulated path.')


🤖 Sentiment-derived stops (score=+0.123):
         Level   Exit fraction   Label
  ---------------------------------------------
        -5.0%          33.3%   Stop 1 (-5%)
       -10.0%          33.3%   Stop 2 (-10%)
       -15.0%          33.3%   Stop 3 (-15%)

Each stop exits that fraction of the REMAINING in-market value.
The rest of the portfolio continues to follow the simulated path.


---
## Phase 2 — Risk Models
### 2a · Download price data & compute returns

In [4]:
raw    = yf.download(TICKERS, start=START, end=END, auto_adjust=True)['Close']
prices = raw.dropna()
rets   = prices.pct_change().dropna()
print(f'Downloaded {len(prices)} days for {len(TICKERS)} tickers.')
prices.tail(3)


[*********************100%***********************]  5 of 5 completed

Downloaded 1257 days for 5 tickers.


Ticker,AAPL,GS,JPM,MSFT,XOM
Date,,,,,
2024-12-26,257.375610,566.623352,235.823593,432.973663,101.351982
2024-12-27,253.967377,561.700256,233.912903,425.482513,101.342461
2024-12-30,250.598892,559.136414,232.118561,419.849365,100.657196


### 2b · Volatility Forecasting (Rolling, EWMA, GARCH)

In [5]:
from arch import arch_model

ticker = 'AAPL'
r      = rets[ticker] * 100

roll_vol = r.rolling(21).std() * np.sqrt(252) / 100
ewma_vol = r.ewm(span=21).std() * np.sqrt(252) / 100

garch     = arch_model(r, vol='Garch', p=1, q=1, rescale=False)
garch_fit = garch.fit(disp='off')
garch_vol = garch_fit.conditional_volatility * np.sqrt(252) / 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=roll_vol.index, y=roll_vol,  name='Rolling 21d'))
fig.add_trace(go.Scatter(x=ewma_vol.index, y=ewma_vol,  name='EWMA'))
fig.add_trace(go.Scatter(x=garch_vol.index, y=garch_vol, name='GARCH(1,1)'))
fig.update_layout(title=f'{ticker} — Annualised Volatility Forecasts',
                  yaxis_tickformat='.0%', template='plotly_dark')
fig.show()
print(f'Latest GARCH vol: {garch_vol.iloc[-1]:.2%}')


Latest GARCH vol: 21.04%


### 2c · VaR & CVaR Engine

In [6]:
from scipy import stats

def compute_risk(returns, confidence_levels=(0.95, 0.99)):
    rows = []
    mu, sigma = returns.mean(), returns.std()
    sim = np.random.normal(mu, sigma, 100_000)
    for cl in confidence_levels:
        a = 1 - cl
        hv  = -np.percentile(returns, a*100)
        hcv = -returns[returns <= -hv].mean()
        pv  = -(mu + stats.norm.ppf(a)*sigma)
        pcv = -(mu - sigma*stats.norm.pdf(stats.norm.ppf(a))/a)
        mv  = -np.percentile(sim, a*100)
        mcv = -sim[sim <= -mv].mean()
        rows.append({'Confidence': f'{cl:.0%}',
                     'Hist VaR': f'{hv:.2%}', 'Hist CVaR': f'{hcv:.2%}',
                     'Param VaR': f'{pv:.2%}', 'Param CVaR': f'{pcv:.2%}',
                     'MC VaR': f'{mv:.2%}', 'MC CVaR': f'{mcv:.2%}'})
    return pd.DataFrame(rows)

display(compute_risk(rets['AAPL']))


,Confidence,Hist VaR,Hist CVaR,Param VaR,Param CVaR,MC VaR,MC CVaR
0,95%,3.01%,4.44%,3.16%,4.00%,3.15%,3.99%
1,99%,5.03%,7.03%,4.53%,5.20%,4.52%,5.19%


### 2d · Regime Detection (Hidden Markov Model)

In [7]:
from hmmlearn.hmm import GaussianHMM

r_arr = rets['AAPL'].values.reshape(-1, 1)
hmm   = GaussianHMM(n_components=3, covariance_type='full',
                    n_iter=200, random_state=42)
hmm.fit(r_arr)
regimes  = hmm.predict(r_arr)
means    = {i: hmm.means_[i][0] for i in range(3)}
ranking  = sorted(means, key=means.get)
labels   = {ranking[0]: 'Bear 🔴', ranking[1]: 'Neutral ⚪', ranking[2]: 'Bull 🟢'}
regime_labels = pd.Series([labels[r] for r in regimes], index=rets.index)

fig = px.scatter(x=rets.index, y=rets['AAPL'], color=regime_labels,
                 color_discrete_map={'Bear 🔴': 'red', 'Neutral ⚪': 'grey', 'Bull 🟢': 'green'},
                 title='AAPL Daily Returns — HMM Regime Detection',
                 template='plotly_dark')
fig.update_traces(marker_size=3)
fig.show()

current_regime = regime_labels.iloc[-1]
print(f'Current regime: {current_regime}')


Current regime: Neutral ⚪


---
## Phase 3 — Portfolio Optimisation (Efficient Frontier)

In [8]:
from scipy.optimize import minimize

mu_annual  = rets.mean() * 252
cov_annual = rets.cov()  * 252
n          = len(TICKERS)

# Sentiment boost: map per-ticker score onto expected returns
for t in TICKERS:
    if t in ticker_sentiment.index:
        mu_annual[t] += ticker_sentiment[t] * 0.01

def portfolio_stats(w):
    ret    = w @ mu_annual
    vol    = np.sqrt(w @ cov_annual @ w)
    sharpe = (ret - RISK_FREE) / vol
    return ret, vol, sharpe

def neg_sharpe(w): return -portfolio_stats(w)[2]

constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1}
bounds      = [(0, 1)] * n
w0          = np.ones(n) / n

opt      = minimize(neg_sharpe, w0, method='SLSQP',
                    bounds=bounds, constraints=constraints)
w_sharpe = opt.x

N_SIM    = 3000
sim_w    = np.random.dirichlet(np.ones(n), N_SIM)
sim_stats = np.array([portfolio_stats(w) for w in sim_w])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sim_stats[:, 1], y=sim_stats[:, 0], mode='markers',
    marker=dict(color=sim_stats[:, 2], colorscale='Viridis', size=4,
                showscale=True, colorbar=dict(title='Sharpe')),
    name='Simulated portfolios'))
r_opt, v_opt, s_opt = portfolio_stats(w_sharpe)
fig.add_trace(go.Scatter(
    x=[v_opt], y=[r_opt], mode='markers+text',
    marker=dict(color='red', size=14, symbol='star'),
    text=['Max Sharpe'], textposition='top right', name='Max Sharpe'))
fig.update_layout(title='Efficient Frontier (sentiment-enhanced returns)',
                  xaxis_title='Volatility', yaxis_title='Return',
                  xaxis_tickformat='.0%', yaxis_tickformat='.0%',
                  template='plotly_dark')
fig.show()

print('\n📌 Optimal weights (Max-Sharpe):')
for t, w in zip(TICKERS, w_sharpe):
    print(f'  {t}: {w:.1%}')
print(f'\n  Return: {r_opt:.2%}  |  Vol: {v_opt:.2%}  |  Sharpe: {s_opt:.2f}')



📌 Optimal weights (Max-Sharpe):
  AAPL: 57.7%
  MSFT: 27.3%
  XOM: 0.0%
  GS: 8.6%
  JPM: 6.4%

  Return: 27.96%  |  Vol: 26.89%  |  Sharpe: 0.85


---
## Phase 4 — Regime-Aware Monte Carlo with Trailing Stops

Each path is simulated via GBM, then the **3-tranche trailing stop engine** runs in real time:
- Tracks the **running peak** of the in-market portion.
- When the drawdown from that peak breaches a stop level, that tranche is **locked in as cash**.
- The remaining position keeps compounding.

The chart shows:
- 200 sample paths, **coloured by which stop was last triggered** (or none).
- Percentile bands for paths **with** and **without** stops.
- Vertical dashed lines at the **average trigger day** for each stop.


In [9]:
# ── Regime parameters ─────────────────────────────────────────────────────
REGIME_PARAMS = {
    'Bull 🟢':    {'drift':  0.12/252, 'vol': 0.12/np.sqrt(252)},
    'Neutral ⚪': {'drift':  0.04/252, 'vol': 0.18/np.sqrt(252)},
    'Bear 🔴':    {'drift': -0.08/252, 'vol': 0.28/np.sqrt(252)},
}

regime    = REGIME_PARAMS.get(current_regime, REGIME_PARAMS['Neutral ⚪'])
drift     = regime['drift'] + portfolio_sentiment * 0.0002  # sentiment nudge
vol       = regime['vol']

np.random.seed(42)

# ── Generate raw GBM paths ────────────────────────────────────────────────
shocks  = np.random.normal(0, 1, (HORIZON, N_PATHS))
log_ret = (drift - 0.5*vol**2) + vol*shocks      # shape (HORIZON, N_PATHS)
daily_rets = np.exp(log_ret)                       # multiplicative returns

# Paths without stops
paths_raw = np.vstack([
    np.full(N_PATHS, float(PORT_VAL)),
    PORT_VAL * np.exp(np.cumsum(log_ret, axis=0))
])  # shape (HORIZON+1, N_PATHS)

# ── Trailing stop engine ──────────────────────────────────────────────────
stops_sorted = sorted(STOPS, key=lambda s: s['level'])  # most negative first

in_market    = np.full(N_PATHS, float(PORT_VAL))
cash         = np.zeros(N_PATHS)
peak         = np.full(N_PATHS, float(PORT_VAL))

triggered    = [np.zeros(N_PATHS, dtype=bool) for _ in stops_sorted]
trigger_day  = [np.full(N_PATHS, -1)          for _ in stops_sorted]

paths_stopped = np.zeros((HORIZON+1, N_PATHS))
paths_stopped[0] = PORT_VAL

for day in range(1, HORIZON+1):
    in_market *= daily_rets[day-1]
    peak       = np.maximum(peak, in_market)
    drawdown   = np.where(peak > 0, (in_market - peak) / peak, 0.0)

    for i, stop in enumerate(stops_sorted):
        fire = (~triggered[i]) & (drawdown <= stop['level'])
        if fire.any():
            locked         = stop['exit_fraction'] * in_market[fire]
            cash[fire]    += locked
            in_market[fire]-= locked
            triggered[i][fire]   = True
            trigger_day[i][fire] = day

    paths_stopped[day] = in_market + cash

# ── Stats ─────────────────────────────────────────────────────────────────
final_raw     = paths_raw[-1]
final_stopped = paths_stopped[-1]

var95_raw     = np.percentile(final_raw,     5)
var95_stopped = np.percentile(final_stopped, 5)

days = np.arange(HORIZON + 1)
pct  = lambda arr, q: np.percentile(arr, q, axis=1)

# ── Plot ──────────────────────────────────────────────────────────────────
# Colour key: which was the LAST stop triggered per path (0=none,1,2,3)
last_stop = np.zeros(N_PATHS, dtype=int)
for i in range(len(stops_sorted)):
    last_stop[triggered[i]] = i + 1

STOP_COLOURS = {0: 'steelblue', 1: '#f0e68c', 2: '#ffa500', 3: '#ff4444'}
STOP_NAMES   = {0: 'No stop hit',
                **{i+1: stops_sorted[i]['label'] for i in range(len(stops_sorted))}}

fig = go.Figure()

sample_idx = np.random.choice(N_PATHS, 200, replace=False)
already_labeled = set()
for i in sample_idx:
    grp  = last_stop[i]
    name = STOP_NAMES[grp]
    show = name not in already_labeled
    already_labeled.add(name)
    fig.add_trace(go.Scatter(
        x=days, y=paths_stopped[:, i],
        line=dict(width=0.5, color=STOP_COLOURS[grp]),
        name=name, legendgroup=name, showlegend=show,
        opacity=0.5
    ))

# Percentile bands — with stops
for q, col, dash, lbl in [(95,'lime','dot','95th (w/ stops)'),
                           (50,'white','solid','Median (w/ stops)'),
                           (5,'red','dot','5th (w/ stops)')]:
    fig.add_trace(go.Scatter(x=days, y=pct(paths_stopped, q),
                             line=dict(color=col, width=2, dash=dash),
                             name=lbl))

# Percentile bands — without stops (dashed grey)
for q, lbl in [(50,'Median (no stops)'), (5,'5th (no stops)')]:
    fig.add_trace(go.Scatter(x=days, y=pct(paths_raw, q),
                             line=dict(color='grey', width=1.5, dash='dash'),
                             name=lbl))

# Vertical lines for average trigger days
for i, stop in enumerate(stops_sorted):
    fired = triggered[i]
    if fired.any():
        avg_day = trigger_day[i][fired].mean()
        pct_hit = fired.mean()
        fig.add_vline(x=avg_day, line_dash='dot',
                      line_color=STOP_COLOURS[i+1], line_width=1.5,
                      annotation_text=f"{stop['label']}<br>avg day {avg_day:.0f} ({pct_hit:.0%})",
                      annotation_font_size=10)

fig.update_layout(
    title=f'Monte Carlo — {N_PATHS:,} paths | Regime: {current_regime} | Sentiment: {portfolio_sentiment:+.3f}',
    xaxis_title='Trading Days', yaxis_title='Portfolio Value ($)',
    template='plotly_dark', height=600
)
fig.show()

# ── Summary ───────────────────────────────────────────────────────────────
print(f'\n📉 1-Year Risk Summary (regime: {current_regime})')
print(f'{"":30}  {"No stops":>14}  {"With stops":>14}')
print('  ' + '─'*62)
for label, raw_q, stp_q in [('Median final value', 50, 50),
                              ('95th pct', 95, 95),
                              ('5th  pct', 5, 5)]:
    print(f'  {label:30}  ${np.percentile(final_raw,raw_q):>12,.0f}  '
          f'${np.percentile(final_stopped,stp_q):>12,.0f}')
print(f'  {"VaR 95%":30}  '
      f'${PORT_VAL-var95_raw:>12,.0f}  ${PORT_VAL-var95_stopped:>12,.0f}')

print('\n📊 Trailing stop trigger rates:')
for i, stop in enumerate(stops_sorted):
    fired = triggered[i]
    if fired.any():
        avg_d = trigger_day[i][fired].mean()
        print(f"  {stop['label']:20}  triggered in {fired.mean():.1%} of paths  "              f"| avg day {avg_d:.0f}")
    else:
        print(f"  {stop['label']:20}  never triggered")



📉 1-Year Risk Summary (regime: Neutral ⚪)
                                      No stops      With stops
  ──────────────────────────────────────────────────────────────
  Median final value              $   1,029,900  $   1,003,673
  95th pct                        $   1,381,435  $   1,178,810
  5th  pct                        $     766,470  $     903,786
  VaR 95%                         $     233,530  $      96,214

📊 Trailing stop trigger rates:
  Stop 3 (-15%)         triggered in 100.0% of paths  | avg day 34
  Stop 2 (-10%)         triggered in 100.0% of paths  | avg day 34
  Stop 1 (-5%)          triggered in 100.0% of paths  | avg day 33


---
## Stress Test — with trailing stops applied

In [10]:
SCENARIOS = {
    'Oil −20%':          {'drift_shock': -0.06/252, 'vol_mult': 1.4},
    'Rates +100 bps':    {'drift_shock': -0.03/252, 'vol_mult': 1.2},
    'Market crash −30%': {'drift_shock': -0.25/252, 'vol_mult': 2.5},
    'Base case':         {'drift_shock':  0.0,       'vol_mult': 1.0},
}

rows = []
N_SC = 5_000
for name, params in SCENARIOS.items():
    d  = drift + params['drift_shock']
    v  = vol   * params['vol_mult']
    lr = (d - 0.5*v**2) + v * np.random.normal(0, 1, (HORIZON, N_SC))
    dr = np.exp(lr)

    # Apply trailing stops
    im   = np.full(N_SC, float(PORT_VAL))
    cs   = np.zeros(N_SC)
    pk   = np.full(N_SC, float(PORT_VAL))
    trig = [np.zeros(N_SC, dtype=bool) for _ in stops_sorted]

    for day in range(HORIZON):
        im *= dr[day]
        pk  = np.maximum(pk, im)
        dd  = np.where(pk > 0, (im - pk) / pk, 0.0)
        for i, stop in enumerate(stops_sorted):
            fire = (~trig[i]) & (dd <= stop['level'])
            if fire.any():
                locked  = stop['exit_fraction'] * im[fire]
                cs[fire] += locked
                im[fire] -= locked
                trig[i][fire] = True

    f = im + cs
    rows.append({
        'Scenario':     name,
        'Median P&L':   f'${np.median(f) - PORT_VAL:>+,.0f}',
        '5th pct P&L':  f'${np.percentile(f,5) - PORT_VAL:>+,.0f}',
        'Prob of loss': f'{(f < PORT_VAL).mean():.1%}',
        'Stop 1 hit':   f'{trig[0].mean():.1%}',
        'Stop 2 hit':   f'{trig[1].mean():.1%}' if len(trig)>1 else 'N/A',
        'Stop 3 hit':   f'{trig[2].mean():.1%}' if len(trig)>2 else 'N/A',
    })

display(pd.DataFrame(rows))


,Scenario,Median P&L,5th pct P&L,Prob of loss,Stop 1 hit,Stop 2 hit,Stop 3 hit
0,Oil −20%,"$-20,349","$-137,576",58.6%,100.0%,100.0%,100.0%
1,Rates +100 bps,"$-7,327","$-118,530",53.6%,100.0%,100.0%,100.0%
2,Market crash −30%,"$-86,290","$-228,028",74.2%,100.0%,100.0%,100.0%
3,Base case,"$+4,299","$-94,581",48.1%,100.0%,100.0%,100.0%
